# Week 4 — Wednesday: Joining Tables — Keys and Relationships

**DATA 202 · Calvin University**

> But Ruth said, "Do not urge me to leave you or to return from following you. For where you go, I will go, and where you lodge, I will lodge. Your people shall be my people, and your God my God." — Ruth 1:16

**Today's theme:** a join is a promise that two records are about the same thing. Ruth's declaration works because *both* sides commit to it. Today: what happens when only *one* side is on record — a plot gardened all season with no paperwork, a registration with no garden yet. `pd.merge()` can't tell relationship from coincidence — it only checks whether keys line up exactly.

**Monday's question, harder version:** one row in `yields`, one row in `registry` — both "about" some plot. How do you tell if it's the **same** plot? → **key column**.

**Setup:** same garden, now **two tables** — Monday's harvest data, and a new registry of who's actually registered. They don't line up as neatly as you'd expect.

**Today's plan (~50 min):**

| Time | Section |
|---|---|
| ~5 min | Load both tables, quick reshape recap |
| ~15 min | Part 1 — Keys and Relational Structure (SLO 04A) |
| ~25 min | Part 2 — The Four Join Types (SLO 04B) |
| ~5 min | Careful with Joining + what's next |

**Same cues as Monday:** 🎯 Predict First · 🙋 Quick Check

---
## Loading Both Tables

In [ ]:
import pandas as pd

yields = pd.read_csv("../../datasets/plot_yields.csv")
registry = pd.read_csv("../../datasets/gardener_registry.csv")

print(yields.shape, registry.shape)
registry.head()

`yields` — Monday's table — has 18 rows, one per harvested plot. `registry` also has 18 rows, but a **completely different subject**: one row per plot *registered* with the coordinator, columns `yields` has never seen (`Garden_Site`, `Years_Gardening`, `Grows_Organic`).

🙋 **Quick Check:** ask Monday's question of each table. One row of `yields` is about ___. One row of `registry` is about ___. Both say "a plot" — specific enough to guarantee row 5 of each is the *same* plot? What single fact would you need to check?

---
## Part 1: Keys and Relational Structure (SLO 04A) · ~15 min

Once two tables can each answer "what is one row about?" with something like *"one plot"*, the only way to connect a row in one to a row in the other is a column that names **which** plot, identically, in both. That column is a **key**.

`yields` and `registry` share exactly one: `Plot_ID`.

* In `yields`, `Plot_ID` is a **primary key** — uniquely identifies each row, the answer to "what is this row about."
* In `registry`, the same `Plot_ID` is a **foreign key** — points back to a row that (hopefully) exists elsewhere: "this row is about that same plot, over there."

A key only works if values match **exactly** — the same string-equality checks you've written all semester. A human answers "what is this about?" with common sense; `pd.merge()` only compares text.

🎯 **Predict First:** both tables have exactly 18 rows — does every `Plot_ID` in `yields` have a match in `registry`? Guess yes/no, then check.

In [ ]:
yields_ids = set(yields["Plot_ID"])
registry_ids = set(registry["Plot_ID"])

print("In yields but not registry:", sorted(yields_ids - registry_ids))
print("In registry but not yields:", sorted(registry_ids - yields_ids))

Two plots (`G17`, `G18`) were harvested all season on a verbal handshake — never formally registered. Two *other* IDs (`G19`, `G20`) are registered for **next** season, no harvest yet. 18 rows each, yet only **16** plots are genuinely in both.

Both tables still answer "what is this row about?" with "a plot" — that hasn't changed. What's missing for `G17`/`G18` is a *partner row on the other side whose key matches, character for character.* The key is exactly where that shared answer holds up or quietly breaks down.

---
### 🔨 Mini-Task A — Confirm It With `.isin()` (~4 min)

1. Filter `yields` to rows whose `Plot_ID` is **not** in `registry["Plot_ID"]` → `yields_only`
2. Filter `registry` to rows whose `Plot_ID` is **not** in `yields["Plot_ID"]` → `registry_only`

Match the set-based check above?

In [ ]:
# Your code here


---
## Part 2: The Four Join Types (SLO 04B) · ~25 min

`pd.merge()` combines two tables on a shared key. `how=` decides what happens to a row that *doesn't* find a match:

| `how=` | Keeps | Unmatched rows get... |
|:---|:---|:---|
| `"inner"` | only rows matched in **both** tables | dropped completely, from both sides |
| `"left"` | every row from the **left** table | `NaN` filled in for the right table's columns |
| `"right"` | every row from the **right** table | `NaN` filled in for the left table's columns |
| `"outer"` | every row from **either** table | `NaN` filled in on whichever side is missing |

🎯 **Predict First:** 16 plots in both, 2 (`G17`, `G18`) in `yields` only, 2 (`G19`, `G20`) in `registry` only. Predict the row count for each join type *before* running.

In [ ]:
inner = pd.merge(yields, registry, on="Plot_ID", how="inner")
inner.shape

In [ ]:
left = pd.merge(yields, registry, on="Plot_ID", how="left")
left.shape

In [ ]:
right = pd.merge(yields, registry, on="Plot_ID", how="right")
right.shape

In [ ]:
outer = pd.merge(yields, registry, on="Plot_ID", how="outer")
outer.shape

16, 18, 18, 20 — four different row counts, same two tables. Right predictions?

🙋 **Quick Check:** which two plots vanish from `inner`? Which columns are `NaN` for `G17`/`G18` in `left`? For `G19`/`G20` in `right`?

In [ ]:
left[left["Garden_Site"].isnull()][["Plot_ID", "Gardener"]]

In [ ]:
right[right["Week1_lbs"].isnull()][["Plot_ID", "Gardener_Name"]]

---
### 🔨 Mini-Task B — Pick the Right Join (~4 min)

Coordinator wants: **every plot actually harvested this season** — registered or not — with registry details filled in where available. Which `how=` gives that in one call? Assign to `harvest_report`, confirm the shape.

In [ ]:
# Your code here
harvest_report = None


---
### 🔨 Task — Organic vs. Non-Organic, Done Right (~8 min)

**Real question:** do organic plots (`Grows_Organic == "Yes"`) harvest more on average than non-organic?

1. **Total each plot's season harvest** — add `Total_lbs` to `yields` (sum `Week1_lbs`...`Week6_lbs`)
2. **Join** — `yields` + `registry` on `Plot_ID`, `how=` that keeps every harvested plot (same choice as Mini-Task B) → `merged`
3. **Group and compare** — group `merged` by `Grows_Organic`, mean of `Total_lbs`. Plots with no `Grows_Organic` value — silently dropped, or their own group? Look closely.

In [ ]:
# Your code here


---
## Careful with Joining

`inner` feels like the "safe" choice — no `NaN`s, everything filled in. But: **`G17` and `G18` really were harvested** — real weeks, real pounds, sitting in `yields` the whole time. An `inner`-only report erases two entire plots' worth of real vegetables, no warning, nothing flagged.

Not a pandas bug — a **choice** (`how="inner"`), made once, early, that quietly shapes every number after it. A plot that's really being harvested but isn't on the official list is the easiest kind of contributor for a report to erase.

→ this week's reading traces *why* a mismatch like `G17` happens, and how a join failure can look identical to genuine absence.

---
## Coming Up

| Topic | What's next |
|---|---|
| This week's reading | Same kind of key mismatch, traced through a *data journey* — who collected it, who cleaned it, who's still missing |
| Practice | Melting, pivoting, joining together, on a new dataset |
| Week 5 | Clustering & Dimensionality Reduction — finding groups the data suggests, instead of ones we choose in advance |